In [4]:
import sys
import os
import torch.nn.functional as F
project_root = os.path.abspath(os.path.join(os.getcwd(),'..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.processor.word_process import sudachi_tokenize
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
import evaluate

In [5]:
# データの読み込み
df = pd.read_csv('../data/raw/edinet_analysis_data_listed_only.csv')
master_df = pd.read_csv('../data/raw/industry_master_33_refined.csv')

# 業種名 -> ID の辞書作成
label2id = {name: i for i, name in enumerate(master_df['業種名'].unique())}
id2label = {i: name for name, i in label2id.items()}

# 学習用データの整形
# E5のルールに従い "query: " を付与
df['text'] = "query: " + df['business_description']
df['label'] = df['33業種区分'].map(label2id)

# 訓練データと検証データに分割（8:2）
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

In [6]:
model_name = "intfloat/multilingual-e5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 分類ヘッド付きモデルのロード
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 574.59it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
class EdinetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EdinetDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer)
val_dataset = EdinetDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer)

In [13]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="../models/results",
    num_train_epochs=3,              # 3エポック程度から開始
    per_device_train_batch_size=8,   # GPUメモリに応じて調整
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    eval_strategy="epoch",      # 毎エポックごとに検証
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 学習開始！
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/home/manaty/company-classification-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.555637,0.567308
2,2.268669,1.035975,0.690934
3,1.005940,0.903616,0.743132


Writing model shards: 100%|███████████████████████| 1/1 [00:01<00:00,  1.19s/it]
/home/manaty/company-classification-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|███████████████████████| 1/1 [00:01<00:00,  1.03s/it]
/home/manaty/company-classification-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|███████████████████████| 1/1 [00:01<00:00,  1.07s/it]


TrainOutput(global_step=1092, training_loss=1.563239296714028, metrics={'train_runtime': 14673.2887, 'train_samples_per_second': 0.595, 'train_steps_per_second': 0.074, 'total_flos': 2296809288078336.0, 'train_loss': 1.563239296714028, 'epoch': 3.0})

In [14]:
output_dir = "../models/e5_edinet_finetuned"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"【大成功】モデルが {output_dir} に保存されました！")

Writing model shards: 100%|███████████████████████| 1/1 [00:07<00:00,  7.88s/it]


【大成功】モデルが ../models/e5_edinet_finetuned に保存されました！
